In [ ]:
# Required Libraries
import yaml
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from enum import Enum

## 1. Path Configuration

In [ ]:
@dataclass
class PathConfig:
    """
    Configuration for project paths.
    Automatically creates directories if they don't exist.
    """
    project_root: Path
    data_dir: Path = None
    artifacts_dir: Path = None
    reports_dir: Path = None
    memory_dir: Path = None
    
    def __post_init__(self):
        # Set default paths relative to project root
        if self.data_dir is None:
            self.data_dir = self.project_root / "data"
        if self.artifacts_dir is None:
            self.artifacts_dir = self.project_root / "artifacts"
        if self.reports_dir is None:
            self.reports_dir = self.project_root / "reports"
        if self.memory_dir is None:
            self.memory_dir = self.project_root / "memory"
        
        # Create directories
        for dir_path in [self.data_dir, self.artifacts_dir, self.reports_dir, self.memory_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
    
    def __repr__(self):
        return f"""PathConfig(
  project_root: {self.project_root}
  data_dir: {self.data_dir}
  artifacts_dir: {self.artifacts_dir}
  reports_dir: {self.reports_dir}
  memory_dir: {self.memory_dir}
)"""

In [ ]:
# Test path configuration
paths = PathConfig(project_root=Path("..").resolve())
print(paths)

## 2. Dataset Registry

In [ ]:
class SourceType(Enum):
    """Type of data source."""
    LOCAL = "local"
    URL = "url"
    DATABASE = "database"


@dataclass
class DatasetInfo:
    """Information about a registered dataset."""
    name: str
    uri: str
    source_type: SourceType
    format: str = "csv"
    description: str = ""
    target_column: Optional[str] = None
    problem_type: Optional[str] = None  # classification, regression, etc.


class DatasetRegistry:
    """
    Registry for managing dataset configurations.
    Loads from YAML file for easy configuration.
    """
    
    def __init__(self):
        self.datasets: Dict[str, DatasetInfo] = {}
    
    def load_from_yaml(self, yaml_path: str):
        """Load datasets from a YAML file."""
        path = Path(yaml_path)
        if not path.exists():
            print(f"⚠️ YAML file not found: {path}")
            return
        
        with open(path, 'r') as f:
            data = yaml.safe_load(f)
        
        for name, config in data.items():
            source = config.get('source', '')
            
            # Determine source type
            if source.startswith('http'):
                source_type = SourceType.URL
                uri = source
            elif source.startswith('local:'):
                source_type = SourceType.LOCAL
                uri = source.replace('local:', '')
            else:
                source_type = SourceType.LOCAL
                uri = source
            
            self.datasets[name] = DatasetInfo(
                name=name,
                uri=uri,
                source_type=source_type,
                format=config.get('format', 'csv'),
                description=config.get('description', ''),
                target_column=config.get('target_column'),
                problem_type=config.get('problem_type')
            )
        
        print(f"✅ Loaded {len(self.datasets)} datasets from {yaml_path}")
    
    def register(self, name: str, uri: str, source_type: SourceType,
                 format: str = "csv", description: str = "",
                 target_column: str = None, problem_type: str = None):
        """Register a dataset manually."""
        self.datasets[name] = DatasetInfo(
            name=name,
            uri=uri,
            source_type=source_type,
            format=format,
            description=description,
            target_column=target_column,
            problem_type=problem_type
        )
        print(f"✅ Registered: {name}")
    
    def get(self, name: str) -> Optional[DatasetInfo]:
        """Get dataset info by name."""
        return self.datasets.get(name)
    
    def list_datasets(self) -> List[str]:
        """List all registered dataset names."""
        return list(self.datasets.keys())
    
    def describe(self, name: str) -> str:
        """Get a description of a dataset."""
        info = self.get(name)
        if info is None:
            return f"Dataset '{name}' not found"
        
        return f"""
📊 Dataset: {info.name}
   Source: {info.source_type.value} - {info.uri}
   Format: {info.format}
   Description: {info.description}
   Target: {info.target_column or 'Not specified'}
   Problem: {info.problem_type or 'Not specified'}
"""

## 3. Test Dataset Registry

In [ ]:
# Create registry
registry = DatasetRegistry()

# Load from YAML
registry.load_from_yaml("../datasets.yaml")

In [ ]:
# List all datasets
print("📋 Registered Datasets:")
for name in registry.list_datasets():
    print(registry.describe(name))

In [ ]:
# Register a new dataset manually
registry.register(
    name="iris",
    uri="https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv",
    source_type=SourceType.URL,
    format="csv",
    description="Classic Iris flower dataset",
    target_column="species",
    problem_type="classification"
)

print(registry.describe("iris"))

## 4. Complete Configuration Class

In [ ]:
@dataclass
class Config:
    """
    Complete configuration for the data science system.
    Combines paths, registry, and other settings.
    """
    paths: PathConfig
    registry: DatasetRegistry = field(default_factory=DatasetRegistry)
    verbose: bool = True
    max_rows_display: int = 10
    
    @classmethod
    def from_project_root(cls, project_root: str = "."):
        """Create config from project root directory."""
        root = Path(project_root).resolve()
        paths = PathConfig(project_root=root)
        
        config = cls(paths=paths)
        
        # Auto-load datasets.yaml if exists
        yaml_path = root / "datasets.yaml"
        if yaml_path.exists():
            config.registry.load_from_yaml(str(yaml_path))
        
        return config
    
    def summary(self) -> str:
        """Get configuration summary."""
        return f"""
{'='*50}
⚙️ CONFIGURATION SUMMARY
{'='*50}

📁 Paths:
   Project Root: {self.paths.project_root}
   Data: {self.paths.data_dir}
   Artifacts: {self.paths.artifacts_dir}
   Reports: {self.paths.reports_dir}
   Memory: {self.paths.memory_dir}

📊 Datasets: {len(self.registry.list_datasets())} registered
   {', '.join(self.registry.list_datasets())}

🔧 Settings:
   Verbose: {self.verbose}
   Max Rows Display: {self.max_rows_display}
{'='*50}
"""

In [ ]:
# Create complete config
config = Config.from_project_root("..")
print(config.summary())

In [ ]:
# Access dataset info
demo_info = config.registry.get("demo_sales")
if demo_info:
    print(f"Demo Sales URI: {demo_info.uri}")
    print(f"Full path: {config.paths.project_root / demo_info.uri}")

## ✅ Summary

This module provides:

**1. PathConfig**
- Manages project directory structure
- Auto-creates directories

**2. DatasetRegistry**
- Load datasets from YAML
- Register datasets manually
- Store metadata (target, problem type, etc.)

**3. Config**
- Combines paths and registry
- Easy initialization from project root